# 8.3 欧盟人工智能法 / The EU AI Act

按风险分级的监管思路，通用模型提供者的义务，对研究和开放权重模型的例外，以及各项规则的生效时间。中国和美国的情况见 8.4。

Regulation graded by risk, the obligations of providers of general-purpose models, the exceptions for research and open-weight models, and when each set of rules applies. China and the United States are covered in 8.4.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 适用日期与罚款上限 / Dates of application and ceilings of fines

把各项规则的适用日期整理成可以查询的函数，并计算第 99 条规定的罚款上限。表中的日期取自欧盟委员会页面 2026 年 10 月 19 日的版本，以后如有调整，只需要修改这张表。

Turn the dates of application of the rules into functions that can be queried, and compute the ceilings of fines under Article 99. The dates in the table are taken from the version of the European Commission page of 19 October 2026, and only this table needs changing if they are adjusted later.

In [ ]:
from datetime import date

# Dates of application as listed on the European Commission's AI Act page (last update 19 October 2026),
# which reflects the amendments of the Digital Omnibus on AI in force since 27 July 2026.
APPLICATION = {
    "prohibited_practices": date(2025, 2, 2),
    "ai_literacy": date(2025, 2, 2),
    "gpai_obligations": date(2025, 8, 2),
    "transparency_art50": date(2026, 8, 2),
    "high_risk_annex_iii": date(2027, 12, 2),
    "high_risk_annex_i": date(2028, 8, 2),
}


def applies(obligation, on):
    """True if the group of obligations applies on the date `on` (a datetime.date). The first day counts."""
    # TODO 1
    raise NotImplementedError


def in_force(on):
    """The names of all groups of obligations that apply on the date `on`, sorted alphabetically."""
    # TODO 2
    raise NotImplementedError


def days_until(obligation, on):
    """Days from `on` until the obligation starts to apply; 0 if it applies already."""
    # TODO 3
    raise NotImplementedError


def max_fine(kind, turnover_eur, sme=False):
    """The ceiling of an administrative fine under Article 99, in euros.

    kind "prohibited":  35 million or 7% of worldwide annual turnover     (prohibited practices)
    kind "other":       15 million or 3%                                  (most other obligations, including Article 50)
    kind "misleading":  7.5 million or 1%                                 (incorrect information to authorities)
    For an undertaking the HIGHER of the two amounts is the ceiling; for an SME the LOWER one.
    Compute the share as turnover_eur * percent / 100 so that round figures stay exact.
    """
    # TODO 4
    raise NotImplementedError

In [ ]:
# 检查 / checks
from datetime import date

assert applies("prohibited_practices", date(2025, 2, 2)) and not applies("prohibited_practices", date(2025, 2, 1))
assert applies("gpai_obligations", date(2026, 1, 1)) and not applies("high_risk_annex_iii", date(2026, 11, 25))
assert in_force(date(2024, 12, 31)) == []
assert in_force(date(2025, 3, 1)) == ["ai_literacy", "prohibited_practices"]
today = date(2026, 11, 25)
assert in_force(today) == ["ai_literacy", "gpai_obligations", "prohibited_practices", "transparency_art50"]
assert in_force(date(2028, 8, 2)) == sorted(APPLICATION)
assert days_until("high_risk_annex_iii", today) == (date(2027, 12, 2) - today).days == 372
assert days_until("gpai_obligations", today) == 0

assert max_fine("prohibited", 100_000_000) == 35_000_000                 # 7% would be 7 million; the fixed amount is higher
assert max_fine("prohibited", 2_000_000_000) == 140_000_000              # 7% of 2 billion
assert max_fine("other", 2_000_000_000) == 60_000_000 and max_fine("misleading", 2_000_000_000) == 20_000_000
assert max_fine("prohibited", 10_000_000, sme=True) == 700_000           # for an SME the lower amount
assert max_fine("other", 1_000_000_000, sme=True) == 15_000_000

print("m83_timeline: all tests passed")
print(f"  applying on {today}: {', '.join(in_force(today))}")
for o in ("high_risk_annex_iii", "high_risk_annex_i"):
    print(f"  {o}: from {APPLICATION[o]}, in {days_until(o, today)} days")

<details><summary><b>提示 1 / Hint 1</b></summary>

`datetime.date` 对象可以直接比较和相减，`(a - b).days` 给出相差的天数。

`datetime.date` objects can be compared and subtracted directly, and `(a - b).days` gives the difference in days.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

比例部分写成 `turnover_eur * percent / 100`。写成 `0.07 * turnover_eur` 会因浮点误差得到不整的数，这正是 5.1 中讨论过的现象。

Write the share as `turnover_eur * percent / 100`. Writing `0.07 * turnover_eur` gives uneven numbers through floating-point error, the very phenomenon discussed in 5.1.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
from datetime import date

# Dates of application as listed on the European Commission's AI Act page (last update 19 October 2026),
# which reflects the amendments of the Digital Omnibus on AI in force since 27 July 2026.
APPLICATION = {
    "prohibited_practices": date(2025, 2, 2),
    "ai_literacy": date(2025, 2, 2),
    "gpai_obligations": date(2025, 8, 2),
    "transparency_art50": date(2026, 8, 2),
    "high_risk_annex_iii": date(2027, 12, 2),
    "high_risk_annex_i": date(2028, 8, 2),
}


def applies(obligation, on):
    return on >= APPLICATION[obligation]


def in_force(on):
    return sorted(o for o in APPLICATION if applies(o, on))


def days_until(obligation, on):
    return max((APPLICATION[obligation] - on).days, 0)


def max_fine(kind, turnover_eur, sme=False):
    fixed, percent = {"prohibited": (35_000_000, 7), "other": (15_000_000, 3), "misleading": (7_500_000, 1)}[kind]
    share = turnover_eur * percent / 100                       # multiply first: 0.07 * x is not exact in floating point
    return min(fixed, share) if sme else max(fixed, share)
```

</details>

## 2. 模型卡检查清单脚本 / A model-card checklist script

写一个脚本，解析模型卡的元数据和正文，检查它是否声明了许可证和语言，是否包含预期用途、局限、训练数据和评估四个部分。测试先在两份构造的模型卡上检查，再读取 Hub 上几个真实模型的模型卡并打印结果，其中包括本路线练习中使用过的几个模型。这个脚本检查的是相应的部分是否存在，不判断内容是否充分，也不是合规性测试。

Write a script that parses the metadata and body of a model card and checks whether it declares a license and a language and contains the four parts on intended use, limitations, training data and evaluation. The tests first check two constructed model cards and then read the cards of several real models on the Hub and print the result, among them models used in the exercises of this roadmap. The script checks whether the parts exist. It does not judge whether their content is adequate and is not a test of compliance.

需要联网下载模型。/ Downloads a model.

In [ ]:
import re
from huggingface_hub import ModelCard

# A checklist for the documentation of a model. It asks whether a model card states its license and
# language and has sections on intended use, limitations, training data and evaluation: the kinds of
# information that Mitchell et al. proposed for model cards and that downstream users need.
# It checks that a section EXISTS, not that its content is adequate, and it is not a test of legal compliance.

SECTIONS = {
    "intended_use": ["intended use", "uses", "use case"],
    "limitations": ["limitation", "bias", "risk"],
    "training_data": ["training data", "training", "dataset"],
    "evaluation": ["evaluation", "results", "performance"],
}


def parse_card(text):
    """Split a model card (the text of a README.md) into its metadata and its body.

    card = ModelCard(text);  card.data.to_dict() is the YAML metadata,  card.text the Markdown after it.
    Returns (metadata_dict, body).
    """
    # TODO 1
    raise NotImplementedError


def headings(body):
    """All Markdown headings of the body (lines starting with one or more #), lower-cased and stripped."""
    # TODO 2: re.findall(r"^#+\\s*(.+)$", body, flags=re.M)
    raise NotImplementedError


def check_card(text):
    """Returns a dict of six booleans:
      "license", "language":  the metadata field is present and not empty
      one entry per key of SECTIONS:  some heading contains one of the keywords of that section
    """
    # TODO 3
    raise NotImplementedError


def completeness(report):
    """The fraction of checklist items that are satisfied."""
    # TODO 4
    raise NotImplementedError

In [ ]:
# 检查 / checks
from huggingface_hub import ModelCard

FULL = """---
license: apache-2.0
language:
- en
base_model: Qwen/Qwen2.5-0.5B-Instruct
---
# GeoQA-mini

A small model for questions about an invented town.

## Intended uses
Teaching material. Not for decisions about people.

## Limitations and bias
Answers may be wrong; see the evaluation.

## Training data
30 hand-written passages.

### Evaluation results
16 of 20 questions correct with retrieval.
"""
THIN = """---
tags:
- text-generation
---
# my-model

## Quickstart
Run it.
"""

meta, body = parse_card(FULL)
assert meta["license"] == "apache-2.0" and meta["language"] == ["en"] and body.lstrip().startswith("# GeoQA-mini")
assert headings(body) == ["geoqa-mini", "intended uses", "limitations and bias", "training data", "evaluation results"]
full, thin = check_card(FULL), check_card(THIN)
assert full == {"license": True, "language": True, "intended_use": True, "limitations": True, "training_data": True, "evaluation": True}
assert thin == {"license": False, "language": False, "intended_use": False, "limitations": False, "training_data": False, "evaluation": False}
assert completeness(full) == 1.0 and completeness(thin) == 0.0
half = check_card("---\nlicense: mit\n---\n# x\n\n## Evaluation\nok\n\n## Known risks\nsome\n")
assert half["license"] and half["evaluation"] and half["limitations"] and not half["language"] and completeness(half) == 0.5
assert check_card("# no metadata at all\n\n## Training\nx\n")["training_data"] is True

# Real cards from the Hub. Only the old GPT-2 card is asserted; the others are printed, since cards get edited.
gpt2 = check_card(ModelCard.load("openai-community/gpt2").content)
assert gpt2["license"] and gpt2["limitations"] and gpt2["training_data"] and gpt2["evaluation"]
rows = [("openai-community/gpt2", gpt2)]
for repo in ("HuggingFaceTB/SmolLM2-135M", "Qwen/Qwen2.5-0.5B-Instruct", "sentence-transformers/all-MiniLM-L6-v2", "cross-encoder/nli-deberta-v3-small"):
    try:
        rows.append((repo, check_card(ModelCard.load(repo).content)))
    except Exception as e:
        rows.append((repo, None))

print("m83_checklist: all tests passed")
print(f"  {'model card':42s} license language intended limits training eval   score")
for repo, r in rows:
    if r is None:
        print(f"  {repo:42s} (could not be loaded)")
        continue
    mark = lambda b: "yes" if b else " - "
    print(f"  {repo:42s} {mark(r['license']):7s} {mark(r['language']):8s} {mark(r['intended_use']):8s} {mark(r['limitations']):6s} {mark(r['training_data']):8s} {mark(r['evaluation']):5s} {completeness(r):5.0%}")

<details><summary><b>提示 1 / Hint 1</b></summary>

`ModelCard(text)` 会自动分开元数据和正文，分别在 `card.data.to_dict()` 和 `card.text` 中。

`ModelCard(text)` separates metadata and body automatically, available as `card.data.to_dict()` and `card.text`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

判断某个部分是否存在，写成 `any(k in h for h in heads for k in keywords)`。

Whether a part exists can be written as `any(k in h for h in heads for k in keywords)`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import re
from huggingface_hub import ModelCard

SECTIONS = {
    "intended_use": ["intended use", "uses", "use case"],
    "limitations": ["limitation", "bias", "risk"],
    "training_data": ["training data", "training", "dataset"],
    "evaluation": ["evaluation", "results", "performance"],
}


def parse_card(text):
    card = ModelCard(text)
    return card.data.to_dict(), card.text


def headings(body):
    return [h.strip().lower() for h in re.findall(r"^#+\s*(.+)$", body, flags=re.M)]


def check_card(text):
    meta, body = parse_card(text)
    heads = headings(body)
    report = {"license": bool(meta.get("license")), "language": bool(meta.get("language"))}
    for item, keywords in SECTIONS.items():
        report[item] = any(k in h for h in heads for k in keywords)
    return report


def completeness(report):
    return sum(report.values()) / len(report)
```

</details>

## 3. 风险分级情景判断 / Risk-level scenarios

根据本模块正文，对下面每个情景说明它属于哪个风险等级或适用哪项例外，涉及的主体是提供者还是部署者，并给出依据。这些判断用于练习，不构成法律意见。
(a) 一家公司在自己的邮件服务中使用模型过滤垃圾邮件。
(b) 一家网店在网站上提供一个回答售后问题的聊天机器人。
(c) 一家招聘平台开发了一个按匹配程度给求职者排序的系统，并销售给企业。
(d) 一家公司想用摄像头和模型判断员工在工作时的情绪状态。
(e) 一名博士生微调了一个开放权重模型，在自己的电脑上做实验并把结果写进论文。
(f) 同一名博士生所在的学院打算用她的模型自动给期末考试的简答题评分。
(g) 一家公司训练了一个通用模型，以 Apache 2.0 许可证公开了权重、结构说明和使用说明。训练的计算量约为 $10^{23}$ 次浮点运算。

Using the text of this module, state for each scenario below which risk level it belongs to or which exception applies, whether the party is a provider or a deployer, and the basis. These judgements are for practice and are not legal advice.
(a) A company filters spam with a model in its own mail service.
(b) An online shop offers a chatbot on its website that answers after-sales questions.
(c) A recruitment platform develops a system that ranks job applicants by fit and sells it to companies.
(d) A company wants to judge the emotional state of employees at work with cameras and a model.
(e) A doctoral student fine-tunes an open-weight model, runs experiments on her own computer and reports the results in a paper.
(f) The department of the same doctoral student plans to use her model to grade the short answers of final exams automatically.
(g) A company trains a general-purpose model and publishes weights, a description of the architecture and usage instructions under the Apache 2.0 licence. The training compute is about $10^{23}$ floating-point operations.

<details><summary><b>提示 1 / Hint 1</b></summary>

(c) 和 (f) 对照附件三的八个领域。

(c) and (f) should be compared with the eight areas of Annex III.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(g) 分别考虑第 53 条的四项义务和系统性风险的门槛。

(g) Consider the four obligations of Article 53 and the threshold for systemic risk separately.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) 风险极小的一级，没有专门的义务。垃圾邮件过滤是委员会页面给出的这一等级的例子。

(b) 有透明度风险的一级。依据第 50 条，提供者须使用户知道自己在与人工智能系统交互，除非这一点显而易见。网店在自己的权限下使用该系统，是部署者。如果聊天机器人是它自己开发并以自己名义提供的，它同时是提供者。

(c) 高风险。筛选和排序求职者属于附件三中的就业和劳动者管理领域。平台是提供者，购买并使用该系统的企业是部署者。系统对求职者进行画像，因此不能援引第 6 条第 3 款的减免。相应的义务自 2027 年 12 月 2 日起适用。

(d) 属于禁止之列。在工作场所进行情绪识别是被禁止的做法之一，该项禁止自 2025 年 2 月起适用。

(e) 适用研究例外。依据第 2 条，专为科学研究与开发的唯一目的而开发并投入使用的系统和模型不在该法的适用范围之内。她仍然需要遵守所用模型的许可证（见 6.2）以及数据保护等其他法律。

(f) 情况发生了变化。评定学习成果属于附件三中的教育和职业培训领域，是高风险用途，并且这是在真实环境中用于决策，不再是纯粹的研究。学院在自己的权限下使用该系统，是部署者。模型如果由学院或该博士生以自己的名义投入使用，还需要确定谁承担提供者的义务。高风险义务自 2027 年 12 月 2 日起适用，在此之前也应当让所在机构的法务和数据保护部门参与。

(g) 该公司是通用模型的提供者。模型以自由和开源许可证发布并公开了参数、结构信息和使用信息，依据第 53 条第 2 款，技术文档和向下游提供信息两项义务被免除，版权政策和训练内容摘要两项仍须履行。训练计算量 $10^{23}$ 低于第 51 条 $10^{25}$ 的门槛，不被推定具有系统性风险。

---

(a) The level of minimal risk, with no specific obligations. Spam filtering is an example the Commission page gives for this level.

(b) The level of transparency risk. Under Article 50 the provider has to make sure that users know they are interacting with an AI system, unless this is obvious. The shop uses the system under its authority and is a deployer. If it developed the chatbot itself and offers it under its own name, it is a provider as well.

(c) High-risk. Screening and ranking job applicants belongs to the area of employment and workers' management in Annex III. The platform is the provider and the companies that buy and use the system are deployers. The system profiles applicants, so the derogation of Article 6(3) cannot be invoked. The corresponding obligations apply from 2 December 2027.

(d) Prohibited. Emotion recognition in the workplace is one of the prohibited practices, and this prohibition has applied since February 2025.

(e) The research exception applies. Under Article 2, systems and models developed and put into service for the sole purpose of scientific research and development are outside the scope of the Act. She still has to comply with the licence of the model used (see 6.2) and with other law such as data protection.

(f) The situation has changed. Grading learning outcomes belongs to the area of education and vocational training in Annex III and is a high-risk use, and it is a use for decisions in a real environment and no longer pure research. The department uses the system under its authority and is a deployer. If the model is put into service by the department or by the doctoral student under their own name, it also has to be settled who bears the obligations of a provider. The high-risk obligations apply from 2 December 2027, and the legal and data protection offices of the institution should be involved before then as well.

(g) The company is a provider of a general-purpose model. The model is released under a free and open-source licence with parameters, information on the architecture and information on usage made public, so under Article 53(2) the two obligations of technical documentation and information to downstream providers are waived, while the copyright policy and the summary of training content remain required. A training compute of $10^{23}$ is below the threshold of $10^{25}$ in Article 51, and systemic risk is not presumed.

</details>